In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

tz = pytz.timezone('Asia/Shanghai')  # UTC+8 as required

# Load files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to get basic summary
def file_summary(df, filename, key_cmdb='cmdb_id'):
    # ensure timestamp exists and is numeric
    if 'timestamp' in df.columns:
        ts_series = pd.to_numeric(df['timestamp'], errors='coerce')
        min_ts = int(ts_series.min()) if not ts_series.isna().all() else None
        max_ts = int(ts_series.max()) if not ts_series.isna().all() else None
    else:
        min_ts = None
        max_ts = None
    cmdb_vals = sorted(pd.Series(df[key_cmdb].dropna().unique()).astype(str).tolist())
    if len(cmdb_vals) > 50:
        cmdb_vals_limited = cmdb_vals[:50]
    else:
        cmdb_vals_limited = cmdb_vals
    return {
        'filename': filename,
        'row_count': int(df.shape[0]),
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'unique_cmdb_ids_sorted_limited': cmdb_vals_limited,
        'unique_cmdb_count': int(len(cmdb_vals))
    }

# Basic summaries
sum_metric = file_summary(df_metric, 'metric.csv')
sum_trace = file_summary(df_trace, 'trace.csv')
sum_log = file_summary(df_log, 'log.csv')
sum_error = file_summary(df_error, 'error_logs.csv')

# Metric-specific details
if {'cmdb_id','kpi_name'}.issubset(df_metric.columns):
    pairs_metric = df_metric[['cmdb_id','kpi_name']].drop_duplicates()
    metric_pair_count = int(pairs_metric.shape[0])
    kpi_names = sorted(df_metric['kpi_name'].dropna().unique().astype(str).tolist())
    kpi_names_limited = kpi_names[:20]
    # example pairs sorted for determinism
    example_pairs = pairs_metric.sort_values(['cmdb_id','kpi_name']).head(20)
    # convert example pairs to list of tuples for compact display
    example_pairs_list = list(example_pairs.itertuples(index=False, name=None))
else:
    metric_pair_count = 0
    kpi_names_limited = []
    example_pairs_list = []

sum_metric.update({
    'distinct_cmdb_kpi_pairs': metric_pair_count,
    'example_kpi_names_up_to_20': kpi_names_limited,
    'example_cmdb_kpi_pairs_up_to_20': example_pairs_list
})

# Trace-specific details
if {'cmdb_id','trace_name'}.issubset(df_trace.columns):
    pairs_trace = df_trace[['cmdb_id','trace_name']].drop_duplicates()
    trace_pair_count = int(pairs_trace.shape[0])
    trace_names = sorted(df_trace['trace_name'].dropna().unique().astype(str).tolist())[:20]
else:
    trace_pair_count = 0
    trace_names = []

sum_trace.update({
    'distinct_cmdb_trace_pairs': trace_pair_count,
    'example_trace_names_up_to_20': trace_names
})

# Log-specific details
if {'cmdb_id','log_name'}.issubset(df_log.columns):
    pairs_log = df_log[['cmdb_id','log_name']].drop_duplicates()
    log_pair_count = int(pairs_log.shape[0])
    log_names = sorted(df_log['log_name'].dropna().unique().astype(str).tolist())[:20]
else:
    log_pair_count = 0
    log_names = []

sum_log.update({
    'distinct_cmdb_log_pairs': log_pair_count,
    'example_log_names_up_to_20': log_names
})

# Error logs specifics: number distinct cmdb_id and up to 20 example messages
if {'timestamp','cmdb_id','message'}.issubset(df_error.columns):
    distinct_error_cmdb_count = int(df_error['cmdb_id'].nunique())
    df_error_sorted = df_error.sort_values('timestamp')[['timestamp','cmdb_id','message']].head(20).copy()
    # truncate messages to first 200 chars
    df_error_sorted['message_snippet'] = df_error_sorted['message'].astype(str).str.slice(0,200)
    # keep only required columns for compactness
    error_examples = df_error_sorted[['timestamp','cmdb_id','message_snippet']].reset_index(drop=True)
else:
    distinct_error_cmdb_count = int(df_error['cmdb_id'].nunique()) if 'cmdb_id' in df_error.columns else 0
    error_examples = pd.DataFrame(columns=['timestamp','cmdb_id','message_snippet'])

sum_error.update({
    'distinct_cmdb_in_error_logs': distinct_error_cmdb_count,
    'example_error_messages_up_to_20': error_examples
})

# Aggregate results
summaries = {
    'metric.csv': sum_metric,
    'trace.csv': sum_trace,
    'log.csv': sum_log,
    'error_logs.csv': sum_error
}

# Display the compact summaries (variable display as required)
summaries

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files (compact):\n\n"
    "metric.csv:\n"
    "- Rows: 28,268\n"
    "- Timestamp range: 1615091400 .. 1615093200 (seconds)\n"
    "- Unique cmdb_ids: 29 (examples: IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, Tomcat01, apache01, ...)\n    (list truncated)\n    - Distinct (cmdb_id, kpi_name) pairs: 1,709\n    - Example kpi names (up to 20, showing first few): Container-..._CpuPercent, Container-..._MemLimit, Container-..._MemPercent, Container-..._MemUsage, Container-..._NetworkRxBytes, ...\n    - Example (cmdb_id, kpi_name) pairs (up to 20 examples shown in prior output; e.g., (IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax), ...)\n\n"
    "trace.csv:\n"
    "- Rows: 6,864\n"
    "- Timestamp range: 1615091400 .. 1615093140 (seconds)\n"
    "- Unique cmdb_ids: 12 (examples: IG01, IG02, MG01, MG02, Tomcat01..Tomcat04, dockerA1/2, dockerB1/2)\n"
    "- Distinct (cmdb_id, trace_name) pairs: 261\n"
    "- Example trace names (up to 20, showing pattern): trace.from_IG01.duration_mean, trace.from_IG01.duration_p95, trace.from_IG01.row_count, trace.from_IG02.duration_mean, ...\n\n"
    "log.csv:\n"
    "- Rows: 504\n"
    "- Timestamp range: 1615091400 .. 1615093200 (seconds)\n"
    "- Unique cmdb_ids: 7 (Mysql01, Tomcat01..Tomcat04, apache01, apache02)\n"
    "- Distinct (cmdb_id, log_name) pairs: 18\n"
    "- Example log names (up to 20): log.error_count, log.gc_pause_max, log.row_count\n\n"
    "error_logs.csv:\n"
    "- Rows: 0 (no error log entries)\n"
    "- No timestamps, cmdb_ids, or messages present to sample.\n\n"
    "Overall notes:\n"
    "- metric.csv contains the largest volume and widest set of KPIs (1,709 distinct cmdb/kpi pairs across 29 components).\n"
    "- trace.csv contains per-component trace features (261 distinct pairs across 12 components).\n"
    "- log.csv is small and contains a few log-derived features for 7 components.\n"
    "- error_logs.csv is empty for this dataset period (no error messages to inspect).\n"
    "Timestamps shown are integer seconds (UTC+8 context used during analysis)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric.csv': {'filename': 'metric.csv', 'row_count': 28268, 'min_timestamp': 1615091400, 'max_timestamp': 1615093200, 'unique_cmdb_ids_sorted_limited': ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'unique_cmdb_count': 29, 'distinct_cmdb_kpi_pairs': 1709, 'example_kpi_names_up_to_20': ['Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_CpuPercent', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_MemLimit', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_MemPercent', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_MemUsage', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_NetworkRxBytes', 'Container-DOCKER_CONTAINER_2d16f5b2e830--bcou-trace-st-uat-statefulset-1--bcou--UATWKR18_NetworkTxBytes', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_CpuPercent', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_MemLimit', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_MemPercent', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_MemUsage', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_NetworkRxBytes', 'Container-DOCKER_CONTAINER_94eca4f96efe--bcou-role-st-uat-statefulset-1--bcou--UATWKR04_NetworkTxBytes', 'Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_CpuPercent', 'Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_MemLimit', 'Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_MemPercent', 'Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_MemUsage', 'Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_NetworkRxBytes', 'Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_NetworkTxBytes', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_CpuPercent', 'Container-DOCKER_CONTAINER_b30097144a13--bcou-trace-st-uat-statefulset-0--bcou--UATWKR04_MemLimit'], 'example_cmdb_kpi_pairs_up_to_20': [('IG01', 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax'), ('IG01', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage'), ('IG01', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed'), ('IG01', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed'), ('IG01', 'JVM-Operating System_7778_JVM_JVM_CPULoad'), ('IG01', 'JVM-Runtime_7778_JVM_JVM_Uptime'), ('IG01', 'JVM-Threads_7778_JVM_ThreadCount_Threads'), ('IG01', 'OSLinux-CPU_CPU-0_SingleCpuUtil'), ('IG01', 'OSLinux-CPU_CPU-0_SingleCpuidle'), ('IG01', 'OSLinux-CPU_CPU-2_SingleCpuUtil'), ('IG01', 'OSLinux-CPU_CPU-2_SingleCpuidle'), ('IG01', 'OSLinux-CPU_CPU_CPUCpuUtil'), ('IG01', 'OSLinux-CPU_CPU_CPULoad'), ('IG01', 'OSLinux-CPU_CPU_CPUSysTime'), ('IG01', 'OSLinux-CPU_CPU_CPUUserTime'), ('IG01', 'OSLinux-CPU_CPU_CPUWio'), ('IG01', 'OSLinux-CPU_CPU_CPUidleutil'), ('IG01', 'OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace'), ('IG01', 'OSLinux-OSLinux_FILESYSTEM_-_FSCapacity'), ('IG01', 'OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent')]}, 'trace.csv': {'filename': 'trace.csv', 'row_count': 6864, 'min_timestamp': 1615091400, 'max_timestamp': 1615093140, 'unique_cmdb_ids_sorted_limited': ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 'unique_cmdb_count': 12, 'distinct_cmdb_trace_pairs': 261, 'example_trace_names_up_to_20': ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95', 'trace.from_Tomcat02.row_count', 'trace.from_Tomcat03.duration_mean', 'trace.from_Tomcat03.duration_p95']}, 'log.csv': {'filename': 'log.csv', 'row_count': 504, 'min_timestamp': 1615091400, 'max_timestamp': 1615093200, 'unique_cmdb_ids_sorted_limited': ['Mysql01', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'], 'unique_cmdb_count': 7, 'distinct_cmdb_log_pairs': 18, 'example_log_names_up_to_20': ['log.error_count', 'log.gc_pause_max', 'log.row_count']}, 'error_logs.csv': {'filename': 'error_logs.csv', 'row_count': 0, 'min_timestamp': None, 'max_timestamp': None, 'unique_cmdb_ids_sorted_limited': [], 'unique_cmdb_count': 0, 'distinct_cmdb_in_error_logs': 0, 'example_error_messages_up_to_20': Empty DataFrame
Columns: [timestamp, cmdb_id, message_snippet]
Index: []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse df_metric loaded previously in the kernel
# Ensure numeric types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Restrict to provided components
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
df_comp = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# 2) Compute global P95 per (cmdb_id, kpi_name) using ALL rows for that pair
# Only consider rows that have a non-null value and kpi_name
df_comp_valid = df_comp[df_comp['kpi_name'].notna() & df_comp['value'].notna()].copy()
p95_series = df_comp_valid.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value':'global_p95_value'})

# Count distinct pairs considered
distinct_pairs_count = int(p95_df.shape[0])

# 3) Filter to time window March 7, 2021 12:30 to 13:00 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2021,3,7,12,30,0))
end_local   = tz.localize(datetime(2021,3,7,13,0,0))
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

df_window = df_comp_valid[(df_comp_valid['timestamp'].notna()) &
                          (df_comp_valid['timestamp'] >= start_ts) &
                          (df_comp_valid['timestamp'] <= end_ts)].copy()

total_rows_in_window = int(df_window.shape[0])

# 4) For each pair, compute aggregates in the window, using the precomputed global P95
if df_window.shape[0] > 0:
    # Merge global P95 into window rows
    df_w = df_window.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
    # Flag anomalies
    df_w['is_anomaly'] = (df_w['value'] >= df_w['global_p95_value']).astype(int)
    # Group aggregates
    grp = df_w.groupby(['cmdb_id','kpi_name']).agg(
        total_points_in_window = ('value','count'),
        anomaly_count = ('is_anomaly','sum'),
        max_value_in_window = ('value','max'),
        global_P95_value = ('global_p95_value','first')
    ).reset_index()
    # earliest anomaly timestamp
    earliest = (df_w[df_w['is_anomaly']==1]
                .groupby(['cmdb_id','kpi_name'])['timestamp']
                .min()
                .reset_index()
                .rename(columns={'timestamp':'earliest_anomaly_timestamp_in_window'}))
    # Merge earliest
    agg = grp.merge(earliest, on=['cmdb_id','kpi_name'], how='left')
    # Fill missing earliest with None, convert to Python None for compactness
    agg['earliest_anomaly_timestamp_in_window'] = agg['earliest_anomaly_timestamp_in_window'].where(
        pd.notna(agg['earliest_anomaly_timestamp_in_window']), None).astype(object)
    # compute severity
    eps = 1e-12
    agg['severity'] = agg['max_value_in_window'] / (agg['global_P95_value'] + eps)
else:
    # No rows in window -> produce empty agg with same columns
    agg = pd.DataFrame(columns=['cmdb_id','kpi_name','total_points_in_window','anomaly_count',
                                'max_value_in_window','global_P95_value','earliest_anomaly_timestamp_in_window','severity'])

# 5) Return compact table up to top 20 sorted by anomaly_count desc then severity desc
if not agg.empty:
    agg_sorted = agg.sort_values(by=['anomaly_count','severity'], ascending=[False,False])
    # Ensure numeric columns are simple types
    agg_sorted['total_points_in_window'] = agg_sorted['total_points_in_window'].astype(int)
    agg_sorted['anomaly_count'] = agg_sorted['anomaly_count'].astype(int)
    # Round numeric values for compactness
    agg_sorted['max_value_in_window'] = agg_sorted['max_value_in_window'].astype(float)
    agg_sorted['global_P95_value'] = agg_sorted['global_P95_value'].astype(float)
    agg_sorted['severity'] = agg_sorted['severity'].astype(float)
    top20_df = agg_sorted.head(20).reset_index(drop=True)
    # Prepare list of dicts with required fields
    result_list = top20_df[['cmdb_id','kpi_name','total_points_in_window','anomaly_count',
                            'earliest_anomaly_timestamp_in_window','max_value_in_window',
                            'global_P95_value','severity']].to_dict('records')
else:
    result_list = []

# 6) Also return counts
pair_count = distinct_pairs_count
rows_in_window = total_rows_in_window

# Display compact outputs (variables)
result_list, pair_count, rows_in_window

```
Out[2]:
```
Summary (restricted to the provided components list):

- Distinct (cmdb_id, kpi_name) pairs considered: 1,636.
- Total metric rows in the window March 7, 2021 12:30–13:00 (UTC+8): 26,826.

Top findings (up to 20, sorted by anomaly_count then severity). Most top pairs had anomaly_count = 31 (an anomaly at every minute in the 31-minute window) and earliest anomaly timestamp = 1615091400 (which is 2021-03-07 12:30:00 Asia/Shanghai):

- IG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT: points=31, anomalies=31, earliest=1615091400, max=1.0, global_P95=0.0, severity ≈ 1.0e12 (P95 is 0).
- Redis01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT: points=31, anomalies=31, earliest=1615091400, max=1.0, global_P95=0.0, severity ≈ 1.0e12 (P95 is 0).
- IG02 — OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc: points=31, anomalies=31, earliest=1615091400, max=0.4, global_P95=0.3, severity ≈ 1.33.
- Redis02 — redis-Redis_6379_Redis  (keyspace_misses): points=31, anomalies=31, earliest=1615091400, max=21226.0, global_P95=21218.0, severity ≈ 1.00038.
- Several JVM memory KPIs (IG01/IG02/MG01/MG02) and Redis memory KPIs show max == global_P95 (severity = 1.0) and anomaly_count = 31 (every timestamp in the window).
- Tomcat request metrics (Tomcat01..Tomcat04 MaxTimeRequestInfo and ErrorCountRequestInfo) similarly show anomalies at every minute with max == global_P95 (severity ≈ 1.0).

Notes:
- "Anomaly" was defined as value >= the global P95 computed across all historical rows for each (cmdb_id,kpi_name) pair (global P95 computed before window filtering).
- A few pairs have global_P95 == 0 and max > 0, producing extremely large severity values due to division by a very small denominator (an epsilon was used to avoid divide-by-zero).
- Many top pairs have anomaly_count equal to the full number of points in the window (31), indicating persistent values at-or-above the global P95 throughout the window.

The original code execution output of IPython Kernel is also provided below for reference:

([{'cmdb_id': 'IG01', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 1.0, 'global_P95_value': 0.0, 'severity': 1000000000000.0}, {'cmdb_id': 'Redis01', 'kpi_name': 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 1.0, 'global_P95_value': 0.0, 'severity': 1000000000000.0}, {'cmdb_id': 'IG02', 'kpi_name': 'OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_PROCPPCPUPerc', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 0.4, 'global_P95_value': 0.3, 'severity': 1.333333333328889}, {'cmdb_id': 'Redis02', 'kpi_name': 'redis-Redis_6379_Redis  (keyspace_misses)', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 21226.0, 'global_P95_value': 21218.0, 'severity': 1.0003770383636534}, {'cmdb_id': 'IG01', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 2040135680.0, 'global_P95_value': 2040135680.0, 'severity': 1.0}, {'cmdb_id': 'IG02', 'kpi_name': 'JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 2040135680.0, 'global_P95_value': 2040135680.0, 'severity': 1.0}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 2040135680.0, 'global_P95_value': 2040135680.0, 'severity': 1.0}, {'cmdb_id': 'MG01', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 506900576.0, 'global_P95_value': 506900576.0, 'severity': 1.0}, {'cmdb_id': 'MG02', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 2040135680.0, 'global_P95_value': 2040135680.0, 'severity': 1.0}, {'cmdb_id': 'MG02', 'kpi_name': 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 947318120.0, 'global_P95_value': 947318120.0, 'severity': 1.0}, {'cmdb_id': 'Redis01', 'kpi_name': 'redis-Redis_6379_Redis  (maxmemory)', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 4294967296.0, 'global_P95_value': 4294967296.0, 'severity': 1.0}, {'cmdb_id': 'Redis01', 'kpi_name': 'redis-Redis_6379_Redis  (used_memory_peak)', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 42627816.0, 'global_P95_value': 42627816.0, 'severity': 1.0}, {'cmdb_id': 'Redis02', 'kpi_name': 'redis-Redis_6379_Redis  (maxmemory)', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 4294967296.0, 'global_P95_value': 4294967296.0, 'severity': 1.0}, {'cmdb_id': 'Redis02', 'kpi_name': 'redis-Redis_6379_Redis  (used_memory_peak)', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 95706776.0, 'global_P95_value': 95706776.0, 'severity': 1.0}, {'cmdb_id': 'Tomcat01', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 348847.0, 'global_P95_value': 348847.0, 'severity': 1.0}, {'cmdb_id': 'Tomcat02', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 399148.0, 'global_P95_value': 399148.0, 'severity': 1.0}, {'cmdb_id': 'Tomcat03', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 351507.0, 'global_P95_value': 351507.0, 'severity': 1.0}, {'cmdb_id': 'Tomcat04', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRequestInfo', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 361406.0, 'global_P95_value': 361406.0, 'severity': 1.0}, {'cmdb_id': 'Redis01', 'kpi_name': 'redis-Redis_6379_Redis  (keyspace_misses)', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 6938.0, 'global_P95_value': 6938.0, 'severity': 0.9999999999999999}, {'cmdb_id': 'Tomcat02', 'kpi_name': 'Tomcat-Requests_7441-"http-nio-8003"_ErrorCountRequestInfo', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 630.0, 'global_P95_value': 630.0, 'severity': 0.9999999999999983}], 1636, 26826)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse df_trace if available in kernel; otherwise load (safe fallback)
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure correct types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series (no time filtering)
df_trace_valid = df_trace[df_trace['trace_name'].notna() & df_trace['value'].notna()].copy()
p95_series_trace = df_trace_valid.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95)
p95_df_trace = p95_series_trace.reset_index().rename(columns={'value':'global_P95_value'})

# Count distinct pairs
distinct_pairs_trace = int(p95_df_trace.shape[0])

# 2) Define UTC+8 time window March 7, 2021 12:30 to 13:00
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2021,3,7,12,30,0))
end_local   = tz.localize(datetime(2021,3,7,13,0,0))
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# 2) Filter rows to the window
df_trace_window = df_trace_valid[(df_trace_valid['timestamp'] >= start_ts) & (df_trace_valid['timestamp'] <= end_ts)].copy()
total_rows_in_trace_window = int(df_trace_window.shape[0])

# 3) Merge global P95 into window rows (use precomputed p95_df_trace)
if not df_trace_window.empty:
    df_tw = df_trace_window.merge(p95_df_trace, on=['cmdb_id','trace_name'], how='left')
    # mark anomalies where value >= global P95
    df_tw['is_anomaly'] = (df_tw['value'] >= df_tw['global_P95_value']).astype(int)
    # aggregates per pair
    agg_trace = df_tw.groupby(['cmdb_id','trace_name']).agg(
        total_points_in_window = ('value','count'),
        anomaly_count = ('is_anomaly','sum'),
        max_value_in_window = ('value','max'),
        global_P95_value = ('global_P95_value','first')
    ).reset_index()
    # earliest anomaly timestamp
    earliest_trace = (df_tw[df_tw['is_anomaly']==1]
                      .groupby(['cmdb_id','trace_name'])['timestamp']
                      .min()
                      .reset_index()
                      .rename(columns={'timestamp':'earliest_anomaly_timestamp_in_window'}))
    agg_trace = agg_trace.merge(earliest_trace, on=['cmdb_id','trace_name'], how='left')
    agg_trace['earliest_anomaly_timestamp_in_window'] = agg_trace['earliest_anomaly_timestamp_in_window'].where(
        pd.notna(agg_trace['earliest_anomaly_timestamp_in_window']), None
    ).astype(object)
    # severity
    eps = 1e-12
    agg_trace['severity'] = agg_trace['max_value_in_window'] / (agg_trace['global_P95_value'] + eps)
else:
    agg_trace = pd.DataFrame(columns=['cmdb_id','trace_name','total_points_in_window','anomaly_count',
                                      'max_value_in_window','global_P95_value','earliest_anomaly_timestamp_in_window','severity'])

# 4) Prepare top 20 sorted by anomaly_count desc then severity desc
if not agg_trace.empty:
    agg_sorted_trace = agg_trace.sort_values(by=['anomaly_count','severity'], ascending=[False,False])
    # ensure compact numeric types
    agg_sorted_trace['total_points_in_window'] = agg_sorted_trace['total_points_in_window'].astype(int)
    agg_sorted_trace['anomaly_count'] = agg_sorted_trace['anomaly_count'].astype(int)
    agg_sorted_trace['max_value_in_window'] = agg_sorted_trace['max_value_in_window'].astype(float)
    agg_sorted_trace['global_P95_value'] = agg_sorted_trace['global_P95_value'].astype(float)
    agg_sorted_trace['severity'] = agg_sorted_trace['severity'].astype(float)
    top20_trace_df = agg_sorted_trace.head(20).reset_index(drop=True)
    result_list = top20_trace_df[['cmdb_id','trace_name','total_points_in_window','anomaly_count',
                                  'earliest_anomaly_timestamp_in_window','max_value_in_window',
                                  'global_P95_value','severity']].to_dict('records')
else:
    result_list = []

# 5) Outputs: result_list (compact), number of distinct pairs, total rows in window
result_list, distinct_pairs_trace, total_rows_in_trace_window

```
Out[3]:
```
Summary of trace.csv analysis (March 7, 2021 12:30–13:00, UTC+8):

- Distinct (cmdb_id, trace_name) pairs considered: 261.
- Total trace rows inside the window: 6,864.

Top findings (up to 20 pairs, sorted by anomaly_count then severity). Key takeaways from the top results:
- dockerB1 / trace.self.duration_p95: 30 points, 6 anomalies, earliest anomaly 1615091520, max=1.014, global P95=1.013, severity ≈ 1.001.
- dockerA2 / trace.self.duration_p95: 30 points, 6 anomalies, earliest 1615091640, max≈global P95 (1.014), severity ≈ 1.00.
- dockerA1 / trace.self.duration_p95: 30 points, 3 anomalies, earliest 1615091880, max=0.014, global P95=0.01, severity ≈ 1.40.
- MG02 / trace.from_dockerA2.row_count and dockerA2 / trace.to_MG02.row_count: each 30 points, 3 anomalies, earliest 1615092660, max=63, global P95=49, severity ≈ 1.29.
- IG02 / trace.self.row_count: 30 points, 3 anomalies, earliest 1615091520, max=1632, global P95=1564, severity ≈ 1.04.
- Several Tomcat↔MG traces (e.g., MG01.trace.from_Tomcat01.row_count and Tomcat01.trace.to_MG01.row_count): 30 points, 3 anomalies, earliest 1615091400, max=202, global P95=200, severity ≈ 1.01.
- High relative severity examples (small P95 with larger max):
  - dockerB2 / trace.self.duration_p95: severity ≈ 53.3 (max 1.009 vs global P95 ≈ 0.01893).
  - MG02 / trace.from_dockerA2.duration_mean and dockerA2 / trace.to_MG02.duration_mean: severity ≈ 3.20 (max≈0.3427 vs P95≈0.1070).
  - MG02 / trace.from_dockerA1.duration_mean and dockerA1 / trace.to_MG02.duration_mean: severity ≈ 3.04.
- Most reported pairs have relatively small anomaly counts (2–6 anomalies out of ~30 points), not sustained anomalies across the whole window.

Notes on methodology (as used in results):
- Global P95 for each (cmdb_id, trace_name) was computed from the full available series before time filtering.
- Rows in the 12:30–13:00 window were tested for value >= that global P95 to count anomalies.
- Severity = max_value_in_window / (global_P95_value + tiny_eps) (tiny eps used to avoid divide-by-zero).

If you want, I can: (a) expand the list to show all top 20 rows in human-readable form, (b) convert earliest anomaly timestamps to readable local time, or (c) highlight only high-severity (>2x) or high-frequency anomalies. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

([{'cmdb_id': 'dockerB1', 'trace_name': 'trace.self.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 6, 'earliest_anomaly_timestamp_in_window': 1615091520, 'max_value_in_window': 1.014, 'global_P95_value': 1.013, 'severity': 1.0009871668302064}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.self.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 6, 'earliest_anomaly_timestamp_in_window': 1615091640, 'max_value_in_window': 1.014, 'global_P95_value': 1.014, 'severity': 0.9999999999990137}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.self.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615091880, 'max_value_in_window': 0.014, 'global_P95_value': 0.01, 'severity': 1.39999999986}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_dockerA2.row_count', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615092660, 'max_value_in_window': 63.0, 'global_P95_value': 49.0, 'severity': 1.2857142857142594}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.to_MG02.row_count', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615092660, 'max_value_in_window': 63.0, 'global_P95_value': 49.0, 'severity': 1.2857142857142594}, {'cmdb_id': 'IG02', 'trace_name': 'trace.self.row_count', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615091520, 'max_value_in_window': 1632.0, 'global_P95_value': 1564.0, 'severity': 1.0434782608695645}, {'cmdb_id': 'MG01', 'trace_name': 'trace.from_Tomcat01.row_count', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 202.0, 'global_P95_value': 200.0, 'severity': 1.009999999999995}, {'cmdb_id': 'Tomcat01', 'trace_name': 'trace.to_MG01.row_count', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 202.0, 'global_P95_value': 200.0, 'severity': 1.009999999999995}, {'cmdb_id': 'Tomcat02', 'trace_name': 'trace.self.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615092540, 'max_value_in_window': 1.006, 'global_P95_value': 1.006, 'severity': 0.9999999999990059}, {'cmdb_id': 'Tomcat01', 'trace_name': 'trace.self.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615092720, 'max_value_in_window': 1.004, 'global_P95_value': 1.004, 'severity': 0.9999999999990039}, {'cmdb_id': 'dockerB2', 'trace_name': 'trace.self.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615091880, 'max_value_in_window': 1.009, 'global_P95_value': 0.01892999999999999, 'severity': 53.301637609439986}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_dockerA2.duration_mean', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092300, 'max_value_in_window': 0.342667, 'global_P95_value': 0.10703934999999998, 'severity': 3.2013180199319105}, {'cmdb_id': 'dockerA2', 'trace_name': 'trace.to_MG02.duration_mean', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092300, 'max_value_in_window': 0.342667, 'global_P95_value': 0.10703934999999998, 'severity': 3.2013180199319105}, {'cmdb_id': 'MG02', 'trace_name': 'trace.from_dockerA1.duration_mean', 'total_points_in_window': 28, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092360, 'max_value_in_window': 0.306889, 'global_P95_value': 0.10100404999999997, 'severity': 3.038383114310384}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.to_MG02.duration_mean', 'total_points_in_window': 28, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092360, 'max_value_in_window': 0.306889, 'global_P95_value': 0.10100404999999997, 'severity': 3.038383114310384}, {'cmdb_id': 'MG02', 'trace_name': 'trace.to_dockerA1.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092180, 'max_value_in_window': 2.04, 'global_P95_value': 1.19591, 'severity': 1.7058139826561314}, {'cmdb_id': 'dockerA1', 'trace_name': 'trace.from_MG02.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092180, 'max_value_in_window': 2.04, 'global_P95_value': 1.19591, 'severity': 1.7058139826561314}, {'cmdb_id': 'IG02', 'trace_name': 'trace.to_Tomcat01.duration_mean', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092720, 'max_value_in_window': 0.94036, 'global_P95_value': 0.573592, 'severity': 1.6394231439740454}, {'cmdb_id': 'Tomcat01', 'trace_name': 'trace.from_IG02.duration_mean', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092720, 'max_value_in_window': 0.94036, 'global_P95_value': 0.573592, 'severity': 1.6394231439740454}, {'cmdb_id': 'IG02', 'trace_name': 'trace.to_Tomcat01.duration_p95', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092060, 'max_value_in_window': 4.4764, 'global_P95_value': 2.900009999999998, 'severity': 1.5435808842033163}], 261, 6864)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse df_log if present, else load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure correct types
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# 1) Compute global P95 per (cmdb_id, log_name) using the full series (no time filtering)
df_log_valid = df_log[df_log['log_name'].notna() & df_log['value'].notna()].copy()
p95_series_log = df_log_valid.groupby(['cmdb_id','log_name'])['value'].quantile(0.95)
p95_df_log = p95_series_log.reset_index().rename(columns={'value':'global_P95_value'})

# Count distinct pairs considered
pair_count = int(p95_df_log.shape[0])

# 2) Define UTC+8 time window March 7, 2021 12:30 to 13:00
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2021,3,7,12,30,0))
end_local   = tz.localize(datetime(2021,3,7,13,0,0))
start_ts = int(start_local.timestamp())
end_ts = int(end_local.timestamp())

# 2) Filter rows to the window
df_log_window = df_log_valid[(df_log_valid['timestamp'] >= start_ts) & (df_log_valid['timestamp'] <= end_ts)].copy()
rows_in_window = int(df_log_window.shape[0])

# 3) Merge global P95 into window rows and compute aggregates
if not df_log_window.empty:
    df_lw = df_log_window.merge(p95_df_log, on=['cmdb_id','log_name'], how='left')
    df_lw['is_anomaly'] = (df_lw['value'] >= df_lw['global_P95_value']).astype(int)
    agg_log = df_lw.groupby(['cmdb_id','log_name']).agg(
        total_points_in_window = ('value','count'),
        anomaly_count = ('is_anomaly','sum'),
        max_value_in_window = ('value','max'),
        global_P95_value = ('global_P95_value','first')
    ).reset_index()
    earliest_log = (df_lw[df_lw['is_anomaly']==1]
                    .groupby(['cmdb_id','log_name'])['timestamp']
                    .min()
                    .reset_index()
                    .rename(columns={'timestamp':'earliest_anomaly_timestamp_in_window'}))
    agg_log = agg_log.merge(earliest_log, on=['cmdb_id','log_name'], how='left')
    agg_log['earliest_anomaly_timestamp_in_window'] = agg_log['earliest_anomaly_timestamp_in_window'].where(
        pd.notna(agg_log['earliest_anomaly_timestamp_in_window']), None).astype(object)
    eps = 1e-12
    agg_log['severity'] = agg_log['max_value_in_window'] / (agg_log['global_P95_value'] + eps)
else:
    agg_log = pd.DataFrame(columns=['cmdb_id','log_name','total_points_in_window','anomaly_count',
                                    'max_value_in_window','global_P95_value','earliest_anomaly_timestamp_in_window','severity'])

# 4) Prepare top 20 sorted by anomaly_count desc then severity desc
if not agg_log.empty:
    agg_sorted_log = agg_log.sort_values(by=['anomaly_count','severity'], ascending=[False,False])
    agg_sorted_log['total_points_in_window'] = agg_sorted_log['total_points_in_window'].astype(int)
    agg_sorted_log['anomaly_count'] = agg_sorted_log['anomaly_count'].astype(int)
    agg_sorted_log['max_value_in_window'] = agg_sorted_log['max_value_in_window'].astype(float)
    agg_sorted_log['global_P95_value'] = agg_sorted_log['global_P95_value'].astype(float)
    agg_sorted_log['severity'] = agg_sorted_log['severity'].astype(float)
    top20_log_df = agg_sorted_log.head(20).reset_index(drop=True)
    result_list = top20_log_df[['cmdb_id','log_name','total_points_in_window','anomaly_count',
                                'earliest_anomaly_timestamp_in_window','max_value_in_window',
                                'global_P95_value','severity']].to_dict('records')
else:
    result_list = []

# Final compact outputs (variable display)
result_list, pair_count, rows_in_window

```
Out[4]:
```
Summary of log.csv analysis for March 7, 2021 12:30–13:00 (UTC+8):

- Distinct (cmdb_id, log_name) pairs considered: 18.
- Total log rows in the window: 504.

Top findings (compact highlights):
- log.error_count (Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, apache02): each had 31 points in the window and anomaly_count = 31. Earliest anomaly timestamp = 1615091400 (2021-03-07 12:30:00 UTC+8). max_value = 0.0, global P95 = 0.0, severity = 0.0.
  - Note: global P95 = 0 causes every non-negative value (including zeros) to be counted as an "anomaly" under the >= P95 rule; these zero-valued series should be interpreted carefully (this is a threshold artifact).
- Mysql01 log.error_count: 24 points, anomaly_count = 24, earliest anomaly 1615091520 (2021-03-07 12:32:00), max = 0.0, global P95 = 0.0.
- log.gc_pause_max (Tomcat03, Tomcat02, Tomcat01, Tomcat04): reported anomalous peaks:
  - Tomcat03: 18 points, anomaly_count = 8, earliest 1615091880 (12:38), max = 0.08, global P95 = 0.08, severity ≈ 1.00.
  - Tomcat02: 30 points, anomaly_count = 2, earliest 1615092300 (12:45), max = 0.12, global P95 ≈ 0.1055, severity ≈ 1.14.
  - Tomcat01: 18 points, anomaly_count = 2, earliest 1615092060 (12:41), max = 0.11, global P95 = 0.11.
  - Tomcat04: 18 points, anomaly_count = 1, earliest 1615092420 (12:47), max = 0.11, global P95 ≈ 0.1015, severity ≈ 1.08.
- log.row_count anomalies (examples):
  - Tomcat03: 31 points, anomaly_count = 3, earliest 1615092660 (12:51), max = 425, global P95 = 394, severity ≈ 1.08.
  - Mysql01: 24 points, anomaly_count = 2, earliest 1615091520 (12:32), max = 5, global P95 ≈ 3.85, severity ≈ 1.30.
  - Tomcat02 / Tomcat04 / Tomcat01 / apache01 / apache02 also show small counts of anomalies (anomaly_count = 2) with severity in the ~1.03–1.09 range.

Interpretation notes:
- Many "anomalies" on log.error_count are due to global P95 = 0 (so value >= 0 flags everything). Treat those results as threshold artifacts rather than meaningful error spikes.
- Genuine actionable anomalies appear in row_count and gc_pause_max series where max > P95 by a non-trivial factor (severity > 1.05 in some cases).
- Earliest anomaly timestamps are within the window (examples given in local times above).

If you want, I can: (a) provide the full top-20 compact list converted to readable timestamps, (b) filter out series with global P95 = 0 before reporting, or (c) flag only pairs with severity > 1.2 or anomaly_count > 5. Which would you like?

The original code execution output of IPython Kernel is also provided below for reference:

([{'cmdb_id': 'Tomcat01', 'log_name': 'log.error_count', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 0.0, 'global_P95_value': 0.0, 'severity': 0.0}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.error_count', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 0.0, 'global_P95_value': 0.0, 'severity': 0.0}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.error_count', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 0.0, 'global_P95_value': 0.0, 'severity': 0.0}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.error_count', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 0.0, 'global_P95_value': 0.0, 'severity': 0.0}, {'cmdb_id': 'apache01', 'log_name': 'log.error_count', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 0.0, 'global_P95_value': 0.0, 'severity': 0.0}, {'cmdb_id': 'apache02', 'log_name': 'log.error_count', 'total_points_in_window': 31, 'anomaly_count': 31, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 0.0, 'global_P95_value': 0.0, 'severity': 0.0}, {'cmdb_id': 'Mysql01', 'log_name': 'log.error_count', 'total_points_in_window': 24, 'anomaly_count': 24, 'earliest_anomaly_timestamp_in_window': 1615091520, 'max_value_in_window': 0.0, 'global_P95_value': 0.0, 'severity': 0.0}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.gc_pause_max', 'total_points_in_window': 18, 'anomaly_count': 8, 'earliest_anomaly_timestamp_in_window': 1615091880, 'max_value_in_window': 0.08, 'global_P95_value': 0.08, 'severity': 0.9999999999874999}, {'cmdb_id': 'Tomcat03', 'log_name': 'log.row_count', 'total_points_in_window': 31, 'anomaly_count': 3, 'earliest_anomaly_timestamp_in_window': 1615092660, 'max_value_in_window': 425.0, 'global_P95_value': 394.0, 'severity': 1.0786802030456826}, {'cmdb_id': 'Mysql01', 'log_name': 'log.row_count', 'total_points_in_window': 24, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615091520, 'max_value_in_window': 5.0, 'global_P95_value': 3.849999999999998, 'severity': 1.298701298700962}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.gc_pause_max', 'total_points_in_window': 30, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092300, 'max_value_in_window': 0.12, 'global_P95_value': 0.10549999999999997, 'severity': 1.1374407582830577}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.row_count', 'total_points_in_window': 31, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092720, 'max_value_in_window': 430.0, 'global_P95_value': 393.0, 'severity': 1.0941475826971983}, {'cmdb_id': 'Tomcat02', 'log_name': 'log.row_count', 'total_points_in_window': 31, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615091520, 'max_value_in_window': 412.0, 'global_P95_value': 383.5, 'severity': 1.0743155149934782}, {'cmdb_id': 'apache02', 'log_name': 'log.row_count', 'total_points_in_window': 31, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092960, 'max_value_in_window': 856.0, 'global_P95_value': 820.0, 'severity': 1.043902439024389}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.row_count', 'total_points_in_window': 31, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615091400, 'max_value_in_window': 401.0, 'global_P95_value': 387.0, 'severity': 1.0361757105943126}, {'cmdb_id': 'apache01', 'log_name': 'log.row_count', 'total_points_in_window': 31, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092660, 'max_value_in_window': 732.0, 'global_P95_value': 727.0, 'severity': 1.0068775790921582}, {'cmdb_id': 'Tomcat01', 'log_name': 'log.gc_pause_max', 'total_points_in_window': 18, 'anomaly_count': 2, 'earliest_anomaly_timestamp_in_window': 1615092060, 'max_value_in_window': 0.11, 'global_P95_value': 0.11, 'severity': 0.999999999990909}, {'cmdb_id': 'Tomcat04', 'log_name': 'log.gc_pause_max', 'total_points_in_window': 18, 'anomaly_count': 1, 'earliest_anomaly_timestamp_in_window': 1615092420, 'max_value_in_window': 0.11, 'global_P95_value': 0.10149999999999999, 'severity': 1.0837438423538548}], 18, 504)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse existing df_metric from kernel
# Ensure types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Allowed components and target KPI
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
target_kpi = 'OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT'

# Time window (UTC+8) March 7, 2021 12:30 to 13:00
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(datetime(2021,3,7,12,30,0)).timestamp())
end_ts = int(tz.localize(datetime(2021,3,7,13,0,0)).timestamp())

# Filter to components and KPI
df_kpi_all = df_metric[
    (df_metric['cmdb_id'].isin(components)) &
    (df_metric['kpi_name'] == target_kpi)
].copy()

# Filter to window
df_kpi_win = df_kpi_all[
    (df_kpi_all['timestamp'] >= start_ts) &
    (df_kpi_all['timestamp'] <= end_ts)
].copy()

# Compute aggregates per cmdb_id
if not df_kpi_win.empty:
    # flag fin-wait (value > 0)
    df_kpi_win['is_fin_wait'] = (df_kpi_win['value'] > 0).astype(int)
    grp = df_kpi_win.groupby('cmdb_id').agg(
        total_points_in_window = ('value','count'),
        fin_wait_count = ('is_fin_wait','sum'),
        max_value_in_window = ('value','max')
    ).reset_index()
    # earliest timestamp where value > 0
    earliest = (df_kpi_win[df_kpi_win['is_fin_wait'] == 1]
                .groupby('cmdb_id')['timestamp']
                .min()
                .reset_index()
                .rename(columns={'timestamp':'earliest_fin_wait_ts'}))
    result_df = grp.merge(earliest, on='cmdb_id', how='left')
    # Keep only rows where fin_wait_count > 0
    result_df = result_df[result_df['fin_wait_count'] > 0].copy()
    # Ensure types and ordering: earliest should be int (or kept as NA if none)
    result_df['earliest_fin_wait_ts'] = result_df['earliest_fin_wait_ts'].astype('Int64')
    # Sort by earliest ascending then fin_wait_count desc
    result_df = result_df.sort_values(by=['earliest_fin_wait_ts','fin_wait_count'], ascending=[True, False])
    # Select and order columns as requested
    result_df = result_df[['cmdb_id','total_points_in_window','fin_wait_count','earliest_fin_wait_ts','max_value_in_window']].reset_index(drop=True)
else:
    # empty result
    result_df = pd.DataFrame(columns=['cmdb_id','total_points_in_window','fin_wait_count','earliest_fin_wait_ts','max_value_in_window'])

# Return compact table limited to top 20 rows
result_df.head(20)

```
Out[5]:
```
Summary (metric KPI = OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT; window = 2021-03-07 12:30–13:00 UTC+8):

- Only four components had any FIN-WAIT > 0 in the window. Each saw a single non-zero measurement:
  - MG01 — total points in window: 22; fin_wait_count: 1; earliest fin-wait: 2021-03-07 12:31:00; max value: 1.0
  - IG01 — total points in window: 31; fin_wait_count: 1; earliest fin-wait: 2021-03-07 12:43:00; max value: 1.0
  - Redis01 — total points in window: 31; fin_wait_count: 1; earliest fin-wait: 2021-03-07 12:53:00; max value: 1.0
  - MG02 — total points in window: 21; fin_wait_count: 1; earliest fin-wait: 2021-03-07 12:57:00; max value: 1.0

Notes:
- total_points_in_window is the count of metric rows present in the window for that cmdb_id/KPI (some components have fewer than the full 31 minutes).
- fin_wait_count > 0 occurs only once per listed component; all non-zero values are 1.0.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_points_in_window  fin_wait_count  earliest_fin_wait_ts  max_value_in_window
0     MG01                      22               1            1615091460                  1.0
1     IG01                      31               1            1615092180                  1.0
2  Redis01                      31               1            1615092780                  1.0
3     MG02                      21               1            1615093020                  1.0```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
from datetime import datetime

# Reuse df_metric from kernel
# Ensure correct types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Allowed components
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# 1) Restrict to allowed components and compute global P95 per (cmdb_id,kpi_name)
df_comp = df_metric[df_metric['cmdb_id'].isin(components)].copy()
df_comp_valid = df_comp[df_comp['kpi_name'].notna() & df_comp['value'].notna()].copy()
p95_series = df_comp_valid.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value':'global_P95_value'})

# 2) Time window (UTC+8) March 7, 2021 12:30 to 13:00
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(datetime(2021,3,7,12,30,0)).timestamp())
end_ts   = int(tz.localize(datetime(2021,3,7,13,0,0)).timestamp())

df_window = df_comp_valid[(df_comp_valid['timestamp'] >= start_ts) & (df_comp_valid['timestamp'] <= end_ts)].copy()

# Merge global P95 into window rows
df_w = df_window.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# 3) Mark exceedance if value >= global_P95 * 1.20
df_w['global_P95_value'] = pd.to_numeric(df_w['global_P95_value'], errors='coerce')
df_w['is_exceed'] = (df_w['value'] >= (df_w['global_P95_value'] * 1.20)).astype(int)

# 4) Detect consecutive runs of exceedances per (cmdb_id,kpi_name)
runs = []
eps = 1e-12
group_cols = ['cmdb_id','kpi_name']

for (cmdb,kpi), grp in df_w.groupby(group_cols):
    g = grp[['timestamp','value','is_exceed','global_P95_value']].sort_values('timestamp').reset_index(drop=True)
    # keep only exceedance rows
    exceed = g[g['is_exceed'] == 1].copy()
    if exceed.empty:
        continue
    # compute time diff in seconds
    diff = exceed['timestamp'].diff().fillna(9999).astype('int64')
    # treat consecutive minutes as diff == 60; allow duplicates (diff==0) as same minute contiguous
    # mark new run when diff != 60 and diff != 0
    new_run_mask = ~((diff == 60) | (diff == 0))
    run_id = new_run_mask.cumsum()
    exceed['run_id'] = run_id
    # aggregate per run
    agg = exceed.groupby('run_id').agg(
        run_start_ts = ('timestamp','min'),
        run_end_ts = ('timestamp','max'),
        length_in_minutes = ('timestamp', lambda x: ((int(x.max()) - int(x.min())) // 60) + 1),
        max_value_in_run = ('value','max'),
        global_P95_value = ('global_P95_value','first')
    ).reset_index(drop=True)
    # compute severity
    agg['severity'] = agg['max_value_in_run'] / (agg['global_P95_value'] + eps)
    # attach identifiers
    agg['cmdb_id'] = cmdb
    agg['kpi_name'] = kpi
    # reorder columns
    agg = agg[['cmdb_id','kpi_name','run_start_ts','run_end_ts','length_in_minutes','max_value_in_run','global_P95_value','severity']]
    runs.append(agg)

# combine all runs
if runs:
    runs_df = pd.concat(runs, ignore_index=True)
else:
    runs_df = pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','length_in_minutes',
                                    'max_value_in_run','global_P95_value','severity'])

# 5) Return top 20 runs sorted by length desc then severity desc, and total number of runs
if not runs_df.empty:
    runs_df['length_in_minutes'] = runs_df['length_in_minutes'].astype(int)
    runs_df['max_value_in_run'] = runs_df['max_value_in_run'].astype(float)
    runs_df['global_P95_value'] = runs_df['global_P95_value'].astype(float)
    runs_df['severity'] = runs_df['severity'].astype(float)
    runs_sorted = runs_df.sort_values(by=['length_in_minutes','severity'], ascending=[False,False]).reset_index(drop=True)
    top20_runs = runs_sorted.head(20)
else:
    top20_runs = runs_df

total_runs = int(runs_df.shape[0])

# Display compact outputs
top20_runs, total_runs

```
Out[6]:
```
Summary of exceedance runs (threshold = value >= 1.20 * global P95; window = 2021-03-07 12:30–13:00 UTC+8):

- Total runs detected across all (cmdb_id, kpi_name) pairs: 3,696.

Top runs (compact highlights, sorted by run length then severity):
1. IG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
   - run_start: 1615091400 (2021-03-07 12:30:00)
   - run_end: 1615093200 (2021-03-07 13:00:00)
   - length: 31 minutes (exceedance present every minute)
   - max_value_in_run: 1.0
   - global_P95_value: 0.0
   - severity: very large (artifact due to P95 = 0)
2. Redis01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
   - same 31-min full-window run, max=1.0, global_P95=0.0, similarly large severity (P95 = 0)
3. IG01 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
   - full 31-minute run, max=0.0, global_P95=0.0 (severity = 0) — flagged because value >= 1.2*P95 with P95=0
4. IG02 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT
   - full 31-minute run, max=0.0, global_P95=0.0
5. IG02 — OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT
   - full 31-minute run, max=0.0, global_P95=0.0
... (many additional full-window runs follow; example entries shown in the top results include various OSLinux and redis-related KPIs all covering the full 31-minute window)

Notes and interpretation:
- Many top runs span the entire 31-minute window (length = 31). This means exceedance condition held for every minute in the window for those (cmdb_id, kpi_name) pairs.
- A number of runs are artifacts caused by global P95 = 0. For example the FIN-WAIT series for IG01 and Redis01 have max = 1.0 but P95 = 0, which yields an enormous severity due to division by (near) zero. These should be treated cautiously — a P95 of zero makes the 1.2×P95 threshold effectively zero, so zeros or small values will be flagged as exceedances.
- Several other runs show max = 0 and global_P95 = 0 (severity = 0) but are still counted because value >= 1.2*P95 (0) is true; these are threshold artifacts rather than actionable spikes.
- Total run count is large (3,696), reflecting many KPI-series where the exceedance rule matched repeated points in the window.

If you want, I can:
- filter out runs where global_P95 == 0 to focus on meaningful exceedances, or
- convert run_start/run_end timestamps to human-readable local times and show the top 20 filtered runs. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                           kpi_name  run_start_ts  run_end_ts  length_in_minutes  max_value_in_run  global_P95_value      severity
0      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    1615091400  1615093200                 31               1.0               0.0  1.000000e+12
1   Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    1615091400  1615093200                 31               1.0               0.0  1.000000e+12
2      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
3      IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
4      IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
5   Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
6   Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
7   Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
8   Mysql02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
9   Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
10  Mysql02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
11  Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
12  Redis01              redis-Redis_6379_Redis  (aof_enabled)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
13  Redis01  redis-Redis_6379_Redis  (aof_rewrite_in_progress)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
14  Redis01    redis-Redis_6379_Redis  (aof_rewrite_scheduled)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
15  Redis01          redis-Redis_6379_Redis  (blocked_clients)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
16  Redis01         redis-Redis_6379_Redis  (connected_slaves)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
17  Redis01             redis-Redis_6379_Redis  (evicted_keys)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
18  Redis01         redis-Redis_6379_Redis  (latest_fork_usec)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00
19  Redis01                  redis-Redis_6379_Redis  (loading)    1615091400  1615093200                 31               0.0               0.0  0.000000e+00, 3696)```
```

In [ ]:
# In[7]:

# Reuse previously computed runs_df if available; otherwise recompute runs as in prior step.
try:
    runs_df
except NameError:
    # Recompute runs (same logic as before) to ensure runs_df exists
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                  "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
    df_comp = df_metric[df_metric['cmdb_id'].isin(components)].copy()
    df_comp_valid = df_comp[df_comp['kpi_name'].notna() & df_comp['value'].notna()].copy()
    p95_series = df_comp_valid.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95)
    p95_df = p95_series.reset_index().rename(columns={'value':'global_P95_value'})
    tz = pytz.timezone('Asia/Shanghai')
    start_ts = int(tz.localize(datetime(2021,3,7,12,30,0)).timestamp())
    end_ts   = int(tz.localize(datetime(2021,3,7,13,0,0)).timestamp())
    df_window = df_comp_valid[(df_comp_valid['timestamp'] >= start_ts) & (df_comp_valid['timestamp'] <= end_ts)].copy()
    df_w = df_window.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')
    df_w['global_P95_value'] = pd.to_numeric(df_w['global_P95_value'], errors='coerce')
    df_w['is_exceed'] = (df_w['value'] >= (df_w['global_P95_value'] * 1.20)).astype(int)
    runs = []
    eps = 1e-12
    for (cmdb,kpi), grp in df_w.groupby(['cmdb_id','kpi_name']):
        g = grp[['timestamp','value','is_exceed','global_P95_value']].sort_values('timestamp').reset_index(drop=True)
        exceed = g[g['is_exceed'] == 1].copy()
        if exceed.empty:
            continue
        diff = exceed['timestamp'].diff().fillna(9999).astype('int64')
        new_run_mask = ~((diff == 60) | (diff == 0))
        run_id = new_run_mask.cumsum()
        exceed['run_id'] = run_id
        agg = exceed.groupby('run_id').agg(
            run_start_ts = ('timestamp','min'),
            run_end_ts = ('timestamp','max'),
            length_in_minutes = ('timestamp', lambda x: ((int(x.max()) - int(x.min())) // 60) + 1),
            max_value_in_run = ('value','max'),
            global_P95_value = ('global_P95_value','first')
        ).reset_index(drop=True)
        agg['severity'] = agg['max_value_in_run'] / (agg['global_P95_value'] + eps)
        agg['cmdb_id'] = cmdb
        agg['kpi_name'] = kpi
        agg = agg[['cmdb_id','kpi_name','run_start_ts','run_end_ts','length_in_minutes','max_value_in_run','global_P95_value','severity']]
        runs.append(agg)
    runs_df = pd.concat(runs, ignore_index=True) if runs else pd.DataFrame(columns=['cmdb_id','kpi_name','run_start_ts','run_end_ts','length_in_minutes','max_value_in_run','global_P95_value','severity'])

# Apply filters:
# (a) global_P95_value > 0
# (b) severity >= 1.5
# (c) length_in_minutes >= 3
filtered_runs = runs_df[
    (runs_df['global_P95_value'] > 0) &
    (runs_df['severity'] >= 1.5) &
    (runs_df['length_in_minutes'] >= 3)
].copy()

# Ensure types and sorting
if not filtered_runs.empty:
    filtered_runs['length_in_minutes'] = filtered_runs['length_in_minutes'].astype(int)
    filtered_runs['max_value_in_run'] = filtered_runs['max_value_in_run'].astype(float)
    filtered_runs['global_P95_value'] = filtered_runs['global_P95_value'].astype(float)
    filtered_runs['severity'] = filtered_runs['severity'].astype(float)
    filtered_sorted = filtered_runs.sort_values(by=['length_in_minutes','severity'], ascending=[False,False]).reset_index(drop=True)
    top20_filtered = filtered_sorted.head(20)
else:
    top20_filtered = filtered_runs

total_filtered_count = int(filtered_runs.shape[0])

# Display compact outputs: top20_filtered (table) and total count
top20_filtered[['cmdb_id','kpi_name','run_start_ts','run_end_ts','length_in_minutes','max_value_in_run','global_P95_value','severity']], total_filtered_count

```
Out[7]:
```
Result: no runs matched the filters.

- Filters applied: (a) global_P95_value > 0, (b) severity >= 1.5 (max_value_in_run >= 150% of P95), and (c) length_in_minutes >= 3.
- Number of runs passing all filters: 0.
- Reasoning summary: after excluding series with P95 == 0 and requiring both a >=50% exceedance over P95 and at least 3 consecutive minutes, no exceedance runs remain. Many previously-detected runs were either artifacts with global_P95 == 0, had severity < 1.5, or were shorter than 3 minutes.

If you want, I can (a) relax the severity or length thresholds, (b) show the top runs by severity even if they don't meet 1.5×, or (c) list runs with global_P95 > 0 sorted by severity. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, run_start_ts, run_end_ts, length_in_minutes, max_value_in_run, global_P95_value, severity]
Index: [], 0)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_trace if available
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure correct types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Restrict to allowed components
components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
              "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
df_trace_comp = df_trace[df_trace['cmdb_id'].isin(components)].copy()

# 2) Compute global P95 per (cmdb_id, trace_name) using full series (no time filtering)
df_trace_valid = df_trace_comp[df_trace_comp['trace_name'].notna() & df_trace_comp['value'].notna()].copy()
p95_series = df_trace_valid.groupby(['cmdb_id','trace_name'])['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value':'global_P95_value'})

# 3) Filter rows to the window March 7, 2021 12:30 to 13:00 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(datetime(2021,3,7,12,30,0)).timestamp())
end_ts   = int(tz.localize(datetime(2021,3,7,13,0,0)).timestamp())

df_window = df_trace_valid[(df_trace_valid['timestamp'] >= start_ts) & (df_trace_valid['timestamp'] <= end_ts)].copy()

# Merge global P95 into window rows
df_w = df_window.merge(p95_df, on=['cmdb_id','trace_name'], how='left')

# 4) Per-trace aggregates within window
eps = 1e-12
# mark anomaly where value >= global_P95_value
df_w['is_anomaly'] = (df_w['value'] >= df_w['global_P95_value']).astype(int)

# per (cmdb_id, trace_name) aggregates in window
per_trace = df_w.groupby(['cmdb_id','trace_name']).agg(
    trace_total_points = ('value','count'),
    trace_anomaly_count = ('is_anomaly','sum'),
    trace_earliest_anom_ts = ('timestamp', lambda x: int(x[df_w.loc[x.index,'is_anomaly'].astype(bool)].min()) if (df_w.loc[x.index,'is_anomaly'].sum()>0) else pd.NA),
    trace_max_value = ('value','max'),
    global_P95_value = ('global_P95_value','first')
).reset_index()

# compute severity per trace (max_value / (global_P95 + eps))
per_trace['severity'] = per_trace['trace_max_value'] / (per_trace['global_P95_value'] + eps)

# 4) Aggregate per cmdb_id across its traces
# total_trace_rows_in_window: sum trace_total_points
# total_anomalous_trace_points: sum trace_anomaly_count
# earliest_anomaly_timestamp_in_window: min of trace_earliest_anom_ts (ignore NA)
# max_severity_across_traces and trace info where this occurs (choose highest severity; tie-breaker highest max_value)
def choose_trace_with_max_severity(sub):
    # choose row with max severity; tie-breaker by trace_max_value desc
    idx = sub[['severity','trace_max_value']].astype(float).sort_values(
        by=['severity','trace_max_value'], ascending=[False,False]
    ).index[0]
    row = sub.loc[idx]
    return pd.Series({
        'trace_name_with_max_severity': row['trace_name'],
        'trace_max_value_for_max_sev': row['trace_max_value'],
        'trace_global_P95_for_max_sev': row['global_P95_value'],
        'max_severity_across_traces': row['severity']
    })

agg_per_cmdb = per_trace.groupby('cmdb_id').agg(
    total_trace_rows_in_window = ('trace_total_points','sum'),
    total_anomalous_trace_points = ('trace_anomaly_count','sum'),
    # earliest anomaly ts per cmdb: take min of trace_earliest_anom_ts ignoring NA
    earliest_anomaly_timestamp_in_window = ('trace_earliest_anom_ts', lambda x: int(np.min([v for v in x if pd.notna(v)])) if any(pd.notna(x)) else pd.NA)
).reset_index()

# get trace with max severity per cmdb
trace_max_info = per_trace.groupby('cmdb_id').apply(choose_trace_with_max_severity).reset_index()

# combine
result = agg_per_cmdb.merge(trace_max_info, on='cmdb_id', how='left')

# Normalize earliest_anomaly_timestamp_in_window: convert pandas NA to None
result['earliest_anomaly_timestamp_in_window'] = result['earliest_anomaly_timestamp_in_window'].where(result['earliest_anomaly_timestamp_in_window'].notna(), None)

# 5) Sort rows by earliest_anomaly_timestamp_in_window ascending (None last) then by max_severity desc
# For sorting, replace None with large number
sort_key = result['earliest_anomaly_timestamp_in_window'].apply(lambda x: x if x is not None else 10**18)
result['sort_earliest_key'] = sort_key
result_sorted = result.sort_values(by=['sort_earliest_key','max_severity_across_traces'], ascending=[True,False]).drop(columns=['sort_earliest_key'])

# Limit to top 20 and select requested columns
out_cols = ['cmdb_id','total_trace_rows_in_window','total_anomalous_trace_points',
            'earliest_anomaly_timestamp_in_window','max_severity_across_traces',
            'trace_name_with_max_severity','trace_max_value_for_max_sev','trace_global_P95_for_max_sev']
result_df = result_sorted[out_cols].reset_index(drop=True).head(20)

# Display compact result (pandas DataFrame)
result_df

```
Out[8]:
```
Summary (trace.csv, restricted components; window = 2021-03-07 12:30–13:00 UTC+8):

Per-component aggregates (one row per cmdb_id), showing total trace rows in the window, total anomalous trace points (value >= that trace's global P95), earliest anomaly timestamp (epoch seconds / local time), and the trace that produced the maximum severity (max_value / global_P95) with its max_value and global_P95:

1. Tomcat01
   - total_trace_rows_in_window: 453
   - total_anomalous_trace_points: 35
   - earliest_anomaly_timestamp_in_window: 1615091400 (2021-03-07 12:30:00)
   - max_severity_across_traces: 1.639423
   - trace with max severity: trace.from_IG02.duration_mean (max_value=0.940360, global_P95=0.573592)

2. MG01
   - total_trace_rows_in_window: 1173
   - total_anomalous_trace_points: 82
   - earliest_anomaly_timestamp_in_window: 1615091400 (2021-03-07 12:30:00)
   - max_severity_across_traces: 1.477925
   - trace with max severity: trace.from_Tomcat04.duration_p95 (max_value=3.193250, global_P95=2.160630)

3. Tomcat04
   - total_trace_rows_in_window: 453
   - total_anomalous_trace_points: 33
   - earliest_anomaly_timestamp_in_window: 1615091400 (2021-03-07 12:30:00)
   - max_severity_across_traces: 1.477925
   - trace with max severity: trace.to_MG01.duration_p95 (max_value=3.193250, global_P95=2.160630)

4. IG01
   - total_trace_rows_in_window: 453
   - total_anomalous_trace_points: 33
   - earliest_anomaly_timestamp_in_window: 1615091400 (2021-03-07 12:30:00)
   - max_severity_across_traces: 1.379584
   - trace with max severity: trace.to_Tomcat02.duration_p95 (max_value=4.138000, global_P95=2.999455)

5. IG02
   - total_trace_rows_in_window: 453
   - total_anomalous_trace_points: 34
   - earliest_anomaly_timestamp_in_window: 1615091460 (2021-03-07 12:31:00)
   - max_severity_across_traces: 1.639423
   - trace with max severity: trace.to_Tomcat01.duration_mean (max_value=0.940360, global_P95=0.573592)

6. MG02
   - total_trace_rows_in_window: 1170
   - total_anomalous_trace_points: 82
   - earliest_anomaly_timestamp_in_window: 1615091520 (2021-03-07 12:32:00)
   - max_severity_across_traces: 3.201318
   - trace with max severity: trace.from_dockerA2.duration_mean (max_value=0.342667, global_P95=0.107039)

7. Tomcat02
   - total_trace_rows_in_window: 453
   - total_anomalous_trace_points: 34
   - earliest_anomaly_timestamp_in_window: 1615091520 (2021-03-07 12:32:00)
   - max_severity_across_traces: 1.379584
   - trace with max severity: trace.from_IG01.duration_p95 (max_value=4.138000, global_P95=2.999455)

8. Tomcat03
   - total_trace_rows_in_window: 453
   - total_anomalous_trace_points: 33
   - earliest_anomaly_timestamp_in_window: 1615091580 (2021-03-07 12:33:00)
   - max_severity_across_traces: 1.531291
   - trace with max severity: trace.self.duration_p95 (max_value=0.833650, global_P95=0.544410)

Notes:
- "Anomalous trace points" were counted per-row by comparing each row's value against that trace's global P95 (computed from the full series before window filtering).
- max_severity_across_traces is the largest ratio (max_value_in_window / global_P95) among the component's traces (tiny eps used to avoid div0).
- Results are sorted by earliest anomaly timestamp (earlier first) then by severity. Only the top 8 components had anomalies in the window; the output is limited to 20 rows as requested.

If you want these earliest timestamps converted to other formats or want to filter components by high severity (e.g., severity > 2), I can produce that.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_trace_rows_in_window  total_anomalous_trace_points  earliest_anomaly_timestamp_in_window  max_severity_across_traces       trace_name_with_max_severity  trace_max_value_for_max_sev  trace_global_P95_for_max_sev
0  Tomcat01                         453                            35                            1615091400                    1.639423      trace.from_IG02.duration_mean                     0.940360                      0.573592
1      MG01                        1173                            82                            1615091400                    1.477925   trace.from_Tomcat04.duration_p95                     3.193250                      2.160630
2  Tomcat04                         453                            33                            1615091400                    1.477925         trace.to_MG01.duration_p95                     3.193250                      2.160630
3      IG01                         453                            33                            1615091400                    1.379584     trace.to_Tomcat02.duration_p95                     4.138000                      2.999455
4      IG02                         453                            34                            1615091460                    1.639423    trace.to_Tomcat01.duration_mean                     0.940360                      0.573592
5      MG02                        1170                            82                            1615091520                    3.201318  trace.from_dockerA2.duration_mean                     0.342667                      0.107039
6  Tomcat02                         453                            34                            1615091520                    1.379584       trace.from_IG01.duration_p95                     4.138000                      2.999455
7  Tomcat03                         453                            33                            1615091580                    1.531291            trace.self.duration_p95                     0.833650                      0.544410```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_metric from kernel
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure correct types and columns
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)

# Target component
cmdb = 'Tomcat01'

# 1) List distinct kpi_name values for Tomcat01 (limit up to 50)
tomcat_all = df_metric[df_metric['cmdb_id'] == cmdb].copy()
tomcat_kpis = sorted(tomcat_all['kpi_name'].dropna().unique().tolist())[:50]

# 2) For each Tomcat01 kpi_name compute global P95 using the full available series (no time filtering)
# Only consider rows with non-null value
tomcat_valid = tomcat_all[tomcat_all['value'].notna()].copy()
p95_series = tomcat_valid.groupby('kpi_name')['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value':'global_P95_value'})

# 3) Filter Tomcat01 rows to the window March 7, 2021 12:30 to 13:00 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(datetime(2021,3,7,12,30,0)).timestamp())
end_ts   = int(tz.localize(datetime(2021,3,7,13,0,0)).timestamp())

tomcat_window = tomcat_valid[(tomcat_valid['timestamp'] >= start_ts) & (tomcat_valid['timestamp'] <= end_ts)].copy()

# 4) Merge global P95 into window rows and compute aggregates per kpi_name
df_w = tomcat_window.merge(p95_df, on='kpi_name', how='left')
eps = 1e-12
df_w['is_anomaly'] = (df_w['value'] >= df_w['global_P95_value']).astype(int)

# Aggregation per kpi_name
def earliest_anom(ts_series, is_anom_series):
    if is_anom_series.sum() == 0:
        return None
    return int(ts_series[is_anom_series.astype(bool)].min())

agg = df_w.groupby('kpi_name').agg(
    total_points_in_window = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    global_P95_value = ('global_P95_value','first')
).reset_index()

# compute earliest anomaly timestamp per kpi_name
earliest = df_w[df_w['is_anomaly']==1].groupby('kpi_name')['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp_in_window'})
agg = agg.merge(earliest, on='kpi_name', how='left')
agg['earliest_anomaly_timestamp_in_window'] = agg['earliest_anomaly_timestamp_in_window'].where(agg['earliest_anomaly_timestamp_in_window'].notna(), None)

# severity
agg['severity'] = agg['max_value_in_window'] / (agg['global_P95_value'] + eps)

# 5) Return compact table sorted by anomaly_count desc then severity desc, limit 50
agg_sorted = agg.sort_values(by=['anomaly_count','severity'], ascending=[False,False]).reset_index(drop=True)
tomcat_summary = agg_sorted[['kpi_name','total_points_in_window','anomaly_count','earliest_anomaly_timestamp_in_window','max_value_in_window','global_P95_value','severity']].head(50)

# Display compact results: list of kpis (up to 50) and the summary dataframe (up to 50 rows)
tomcat_kpis, tomcat_summary

```
Out[9]:
```
```python
summary = (
    "Tomcat01 KPI summary (window: 2021-03-07 12:30–13:00 UTC+8):\n\n"
    "- Distinct kpi_name values found (limited to 50): 50 names listed.\n\n"
    "- High-level findings:\n"
    "  * Many Tomcat01 KPIs had anomaly_count = 31 (an anomaly at every minute in the 31-minute window).\n"
    "    Examples: Tomcat-Requests_7441-\"http-nio-8003\"_MaxTimeRequestInfo (max=348,847, P95=348,847, severity=1.0),\n"
    "              Tomcat-Threads_7441-\"http-nio-8003\"_MaxThreads (max=500, P95=500, severity=1.0),\n"
    "              Tomcat-Requests_7441-\"http-nio-8003\"_ErrorCountRequestInfo (max=326, P95=326, severity=1.0).\n\n"
    "  * Several KPI series are zero-valued with global P95 = 0 (e.g., some session counters and network error metrics).\n"
    "    These appear as anomalies by the >= P95 rule but are threshold artifacts and should be interpreted cautiously.\n\n"
    "  * For most KPIs where max_value_in_window equals global_P95_value the computed severity is 1.0, indicating values at the P95 level.\n\n"
    "- Suggested next steps:\n"
    "  * Inspect the specific KPI time series where anomaly_count = 31 but P95 = 0 (to filter out artifacts).\n"
    "  * For KPIs with large absolute values at P95 (e.g., MaxTimeRequestInfo), investigate whether sustained P95-levels are expected.\n\n"
    "If you want, I can now: (a) show the full per-KPI summary rows for Tomcat01, (b) convert earliest anomaly timestamps to readable times, "
    "or (c) filter out P95==0 series and show only true exceedances. Which would you like?"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(['OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-home_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-home_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-home_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-home_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-tmp_FSUsedSpace', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailableSpace', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace', 'OSLinux-OSLinux_FILE_-home-zabbix_DirSizeMB', 'OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_FileSizeMB', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPercentBusy', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvgServ'],                                              kpi_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp_in_window  max_value_in_window  global_P95_value  severity
0   Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                      31             31                            1615091400             348847.0          348847.0       1.0
1   Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31             31                            1615091400                500.0             500.0       1.0
2   Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31             31                            1615091400                326.0             326.0       1.0
3     Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter                      31             31                            1615091400                  0.0               0.0       0.0
4   Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...                      31             31                            1615091400                  0.0               0.0       0.0
..                                                ...                     ...            ...                                   ...                  ...               ...       ...
45  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSUsedS...                       9              9                            1615091460                 15.0              15.0       1.0
46            OSLinux-OSLinux_NETWORK_ens160_NETInErr                       9              9                            1615091460                  0.0               0.0       0.0
47           OSLinux-OSLinux_NETWORK_ens160_NETOutErr                       9              9                            1615091460                  0.0               0.0       0.0
48                       OSLinux-OSLinux_SWAP_SWAP_Si                       9              9                            1615091460                  0.0               0.0       0.0
49                       OSLinux-OSLinux_SWAP_SWAP_So                       9              9                            1615091460                  0.0               0.0       0.0

[50 rows x 7 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np

# Reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure types
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace['cmdb_id'] = df_trace['cmdb_id'].astype(str)
df_trace['trace_name'] = df_trace['trace_name'].astype(str)

# Target component
cmdb = 'Tomcat01'

# 1) Compute global P95 per trace_name for Tomcat01 using full series (no time filter)
df_tomcat_all = df_trace[df_trace['cmdb_id'] == cmdb].copy()
df_tomcat_valid = df_tomcat_all[df_tomcat_all['value'].notna() & df_tomcat_all['trace_name'].notna()].copy()
p95_series = df_tomcat_valid.groupby('trace_name')['value'].quantile(0.95)
p95_df = p95_series.reset_index().rename(columns={'value':'global_P95_value'})

# 2) Filter Tomcat01 rows to the window March 7, 2021 12:30 to 13:00 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(datetime(2021,3,7,12,30,0)).timestamp())
end_ts   = int(tz.localize(datetime(2021,3,7,13,0,0)).timestamp())

df_window = df_tomcat_valid[(df_tomcat_valid['timestamp'] >= start_ts) & (df_tomcat_valid['timestamp'] <= end_ts)].copy()

# 3) Merge global P95 into window rows
df_w = df_window.merge(p95_df, on='trace_name', how='left')

# Mark anomalies where value >= global_P95_value
eps = 1e-12
df_w['is_anomaly'] = (df_w['value'] >= df_w['global_P95_value']).astype(int)

# 4) Per-trace aggregates over the filtered rows
per_trace = df_w.groupby('trace_name').agg(
    total_points_in_window = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    earliest_anomaly_timestamp_in_window = ('timestamp', lambda x: int(x[df_w.loc[x.index,'is_anomaly'].astype(bool)].min()) if (df_w.loc[x.index,'is_anomaly'].sum()>0) else pd.NA),
    max_value_in_window = ('value','max'),
    global_P95_value = ('global_P95_value','first')
).reset_index()

# severity
per_trace['severity'] = per_trace['max_value_in_window'] / (per_trace['global_P95_value'] + eps)

# Normalize earliest timestamp: replace pandas NA with None for readability
per_trace['earliest_anomaly_timestamp_in_window'] = per_trace['earliest_anomaly_timestamp_in_window'].where(per_trace['earliest_anomaly_timestamp_in_window'].notna(), None)

# 5) Sort by earliest_anomaly_timestamp_in_window ascending (None last) then severity desc
# Create sort key where None -> large number
per_trace['sort_earliest'] = per_trace['earliest_anomaly_timestamp_in_window'].apply(lambda x: x if x is not None else 10**18)
per_trace_sorted = per_trace.sort_values(by=['sort_earliest','severity'], ascending=[True,False]).drop(columns=['sort_earliest'])

# Select required columns and limit to top 20
tomcat_trace_summary = per_trace_sorted[
    ['trace_name','total_points_in_window','anomaly_count','earliest_anomaly_timestamp_in_window',
     'max_value_in_window','global_P95_value','severity']
].head(20).reset_index(drop=True)

# Display compact result
tomcat_trace_summary

```
Out[10]:
```
Summary for Tomcat01 traces (window = 2021-03-07 12:30–13:00 UTC+8):

- Number of trace series with reported results: 18 (table limited to top 20).
- Method: global P95 per (Tomcat01, trace_name) was computed from the full series; anomalies = rows with value >= that trace's global P95.

Key aggregated findings (trace_name — points, anomaly_count, earliest anomaly (epoch → local time), max_value, global_P95, severity = max / P95):

- trace.from_IG02.duration_mean — 30 pts, 2 anomalies, earliest 1615092720 → 2021-03-07 12:52:00, max=0.94036, P95=0.573592, severity ≈ 1.639
- trace.from_IG02.duration_p95 — 30 pts, 2 anomalies, earliest 1615092060 → 2021-03-07 12:41:00, max=4.4764, P95=2.90001, severity ≈ 1.544
- trace.self.duration_mean — 30 pts, 2 anomalies, earliest 1615092060 → 12:41:00, max=0.116237, P95=0.085525, severity ≈ 1.359
- trace.to_MG01.duration_p95 — 30 pts, 2 anomalies, earliest 1615092060 → 12:41:00, max=3.961, P95=3.133245, severity ≈ 1.264
- trace.to_MG01.duration_mean — 30 pts, 2 anomalies, earliest 1615092060 → 12:41:00, max=0.705632, P95=0.566277, severity ≈ 1.246
- trace.to_MG02.duration_p95 — 30 pts, 2 anomalies, earliest 1615091520 → 12:32:00, max=3.1358, P95=2.52961, severity ≈ 1.240
- trace.to_MG02.duration_mean — 30 pts, 2 anomalies, earliest 1615091880 → 12:38:00, max=0.601104, P95=0.492779, severity ≈ 1.220
- trace.self.duration_p95 — 30 pts, 3 anomalies, earliest 1615092720 → 12:52:00, max=1.004, P95=1.004, severity = 1.000
- row_count series with earliest anomalies at the window start (12:30):
  - trace.from_IG01.row_count — 30 pts, 2 anomalies, earliest 1615091400 → 12:30:00, max=210, P95=201.3, severity ≈ 1.043
  - trace.self.row_count — 30 pts, 2 anomalies, earliest 12:30:00, max=2744, P95=2657.4, severity ≈ 1.033
  - trace.to_MG01.row_count — 30 pts, 3 anomalies, earliest 12:30:00, max=202, P95=200, severity ≈ 1.010

Other notes:
- Most trace series have modest severity (1.01–1.64), i.e., up to ~1.6× their P95.
- Several traces show earliest anomalies at the start of the window (12:30); others occur later (12:32, 12:38, 12:41, 12:52, 12:54).
- A few single-point traces (e.g., trace.from_root series) appear with 1 point and anomaly_count = 1 (severity = 1.0).

If you want, I can (a) show these rows with human-readable timestamps in a table, (b) filter to severity > 1.3, or (c) plot the time series for a selected trace to inspect the anomaly pattern. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp_in_window  max_value_in_window  global_P95_value  severity
0       trace.from_IG01.row_count                      30              2                            1615091400           210.000000        201.300000  1.043219
1            trace.self.row_count                      30              2                            1615091400          2744.000000       2657.400000  1.032588
2         trace.to_MG01.row_count                      30              3                            1615091400           202.000000        200.000000  1.010000
3      trace.to_MG02.duration_p95                      30              2                            1615091520             3.135800          2.529610  1.239638
4   trace.from_IG01.duration_mean                      30              2                            1615091520             0.663821          0.596279  1.113272
5    trace.from_IG01.duration_p95                      30              2                            1615091520             3.145850          2.901302  1.084289
6       trace.from_IG02.row_count                      30              2                            1615091520           207.000000        200.050000  1.034741
7     trace.to_MG02.duration_mean                      30              2                            1615091880             0.601104          0.492779  1.219825
8    trace.from_IG02.duration_p95                      30              2                            1615092060             4.476400          2.900010  1.543581
9        trace.self.duration_mean                      30              2                            1615092060             0.116237          0.085525  1.359096
10     trace.to_MG01.duration_p95                      30              2                            1615092060             3.961000          3.133245  1.264185
11    trace.to_MG01.duration_mean                      30              2                            1615092060             0.705632          0.566277  1.246089
12      trace.from_root.row_count                       1              1                            1615092600             7.000000          7.000000  1.000000
13   trace.from_root.duration_p95                       1              1                            1615092600             0.021300          0.021300  1.000000
14  trace.from_root.duration_mean                       1              1                            1615092600             0.005000          0.005000  1.000000
15  trace.from_IG02.duration_mean                      30              2                            1615092720             0.940360          0.573592  1.639423
16        trace.self.duration_p95                      30              3                            1615092720             1.004000          1.004000  1.000000
17        trace.to_MG02.row_count                      30              2                            1615092840           201.000000        197.400000  1.018237```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
from datetime import datetime
import numpy as np
import re

# Reuse df_metric from kernel
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure correct types
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric['cmdb_id'] = df_metric['cmdb_id'].astype(str)
df_metric['kpi_name'] = df_metric['kpi_name'].astype(str)

# Target component and keywords (case-insensitive)
cmdb = 'Tomcat01'
keywords = ['cpu', 'jvm', 'thread', 'heap', 'memory', 'gc_pause', 'gc']
# Build regex: ensure longer token gc_pause checked but regex matching is fine since case-insensitive
pattern = '(' + '|'.join(re.escape(k) for k in keywords) + ')'

# 1) Identify matching kpi_name values for Tomcat01 (limit up to 50)
tomcat_all = df_metric[df_metric['cmdb_id'] == cmdb].copy()
matched_mask = tomcat_all['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
matched_kpis = sorted(tomcat_all.loc[matched_mask, 'kpi_name'].dropna().unique().tolist())[:50]

# 2) Compute global P95 per matched kpi_name using the full Tomcat01 series (no time filtering)
tomcat_valid = tomcat_all[tomcat_all['value'].notna()].copy()
p95_df = tomcat_valid[tomcat_valid['kpi_name'].isin(matched_kpis)].groupby('kpi_name', as_index=False)['value'].quantile(0.95).rename(columns={'value':'global_P95_value'})

# 3) Filter Tomcat01 rows to the window March 7, 2021 12:30 to 13:00 (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(tz.localize(datetime(2021,3,7,12,30,0)).timestamp())
end_ts   = int(tz.localize(datetime(2021,3,7,13,0,0)).timestamp())

tomcat_window = tomcat_valid[
    (tomcat_valid['timestamp'] >= start_ts) &
    (tomcat_valid['timestamp'] <= end_ts) &
    (tomcat_valid['kpi_name'].isin(matched_kpis))
].copy()

# 4) Merge global P95 and compute aggregates per kpi_name
df_w = tomcat_window.merge(p95_df, on='kpi_name', how='left')
eps = 1e-12
df_w['is_anomaly'] = (df_w['value'] >= df_w['global_P95_value']).astype(int)

# per-kpi aggregates
agg = df_w.groupby('kpi_name').agg(
    total_points_in_window = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    global_P95_value = ('global_P95_value','first')
).reset_index()

# earliest anomaly timestamp per kpi_name
earliest = df_w[df_w['is_anomaly'] == 1].groupby('kpi_name', as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_timestamp_in_window'})
agg = agg.merge(earliest, on='kpi_name', how='left')
agg['earliest_anomaly_timestamp_in_window'] = agg['earliest_anomaly_timestamp_in_window'].where(agg['earliest_anomaly_timestamp_in_window'].notna(), None)

# severity
agg['severity'] = agg['max_value_in_window'] / (agg['global_P95_value'] + eps)

# 5) Sort by severity desc then anomaly_count desc, limit to top 20
agg_sorted = agg.sort_values(by=['severity','anomaly_count'], ascending=[False,False]).reset_index(drop=True)
result_df = agg_sorted[['kpi_name','total_points_in_window','anomaly_count',
                        'earliest_anomaly_timestamp_in_window','max_value_in_window',
                        'global_P95_value','severity']].head(20)

# Display compact result (DataFrame)
result_df

```
Out[11]:
```
Summary for Tomcat01 (metrics matching keywords cpu, jvm, thread, heap, memory, gc, gc_pause; window 2021-03-07 12:30–13:00 UTC+8):

- Matching KPI examples (up to 50 were scanned), top results shown below.

Top-ranked KPIs by severity (severity = max_value_in_window / global_P95_value):

1. OSLinux-CPU_CPU_CPUWio
   - total_points: 21, anomalies: 2
   - earliest anomaly: 2021-03-07 12:51:00
   - max = 0.5157, global_P95 = 0.2162, severity ≈ 2.385

2. OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_... (zabbix agent CPU %)
   - total_points: 21, anomalies: 21
   - earliest anomaly: 2021-03-07 12:30:00
   - max = 0.4, global_P95 = 0.3, severity = 1.333

3. Tomcat-Threads_..._CurrentThreads... (thread count)
   - total_points: 31, anomalies: 5
   - earliest anomaly: 2021-03-07 12:36:00
   - max = 6.0, global_P95 = 5.0, severity = 1.20

4. OSLinux-CPU_CPU_CPUSysTime
   - total_points: 21, anomalies: 2
   - earliest anomaly: 2021-03-07 12:34:00
   - max = 0.7735, global_P95 = 0.7067, severity ≈ 1.095

5. OSLinux-CPU_CPU-0_SingleCpuUtil
   - total_points: 11, anomalies: 1
   - earliest anomaly: 2021-03-07 12:37:00
   - max = 2.709, global_P95 = 2.50145, severity ≈ 1.083

6. OSLinux-CPU_CPU-3_SingleCpuUtil
   - total_points: 11, anomalies: 1
   - earliest anomaly: 2021-03-07 12:56:00
   - max = 4.4433, global_P95 = 4.2613, severity ≈ 1.043

7. OSLinux-CPU_CPU_CPUCpuUtil
   - total_points: 20, anomalies: 1
   - earliest anomaly: 2021-03-07 12:45:00
   - max = 27.9115, global_P95 = 27.10058, severity ≈ 1.030

8. OSLinux-CPU_CPU_CPULoad
   - total_points: 21, anomalies: 2
   - earliest anomaly: 2021-03-07 12:55:00
   - max = 1.29, global_P95 = 1.26, severity ≈ 1.024

9. OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem
   - total_points: 20, anomalies: 1
   - earliest anomaly: 2021-03-07 12:44:00
   - max = 145.0, global_P95 = 142.15, severity ≈ 1.020

10. Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory
    - total_points: 3, anomalies: 1
    - earliest anomaly: 2021-03-07 12:37:00
    - max = 3.1366e9, global_P95 ≈ 3.0770e9, severity ≈ 1.019

- Other notes:
  - Many matched KPIs show small severities near 1.0 (max ≈ P95), indicating values at or just above the historical P95 rather than large spikes.
  - Some series (e.g., zabbix process metric) had anomalies at every sampled minute in the window (anomaly_count = total_points).
  - Data completeness varies by KPI (total_points ranges from a few to 31).

If you want, I can (a) return the full table for these matched KPIs with readable timestamps, (b) filter to severity > 1.3, or (c) focus on JVM-specific KPIs only. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp_in_window  max_value_in_window  global_P95_value  severity
0                              OSLinux-CPU_CPU_CPUWio                      21              2                            1615092660         5.157000e-01      2.162000e-01  2.385291
1   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                      21             21                            1615091400         4.000000e-01      3.000000e-01  1.333333
2   Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31              5                            1615091760         6.000000e+00      5.000000e+00  1.200000
3                          OSLinux-CPU_CPU_CPUSysTime                      21              2                            1615091640         7.735000e-01      7.067000e-01  1.094524
4                     OSLinux-CPU_CPU-0_SingleCpuUtil                      11              1                            1615091820         2.709000e+00      2.501450e+00  1.082972
5                     OSLinux-CPU_CPU-3_SingleCpuUtil                      11              1                            1615092960         4.443300e+00      4.261300e+00  1.042710
6                          OSLinux-CPU_CPU_CPUCpuUtil                      20              1                            1615092300         2.791150e+01      2.710058e+01  1.029923
7                             OSLinux-CPU_CPU_CPULoad                      21              2                            1615092900         1.290000e+00      1.260000e+00  1.023810
8            OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem                      20              1                            1615092240         1.450000e+02      1.421500e+02  1.020049
9             Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                       3              1                            1615091820         3.136636e+09      3.076959e+09  1.019395
10                    OSLinux-CPU_CPU-2_SingleCpuUtil                      11              1                            1615093140         1.000000e+02      9.961780e+01  1.003837
11            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                       6              1                            1615092060         1.694155e+09      1.687695e+09  1.003828
12  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31              5                            1615091400         3.100000e+02      3.090000e+02  1.003236
13                    OSLinux-CPU_CPU-0_SingleCpuidle                      10              1                            1615091460         9.908560e+01      9.886812e+01  1.002200
14                    OSLinux-CPU_CPU-1_SingleCpuidle                      10              1                            1615091940         9.911950e+01      9.895453e+01  1.001667
15                    OSLinux-CPU_CPU-2_SingleCpuidle                      10              1                            1615091580         9.883590e+01      9.876102e+01  1.000758
16                        OSLinux-CPU_CPU_CPUUserTime                      21              2                            1615091400         2.592050e+01      2.590560e+01  1.000575
17                        OSLinux-CPU_CPU_CPUidleutil                      21              2                            1615091940         7.344120e+01      7.340510e+01  1.000492
18             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem                      20              1                            1615092660         4.086000e+03      4.084100e+03  1.000465
19       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc                      21              3                            1615091520         4.611630e+01      4.610350e+01  1.000278```
```